In [ ]:
import tensorflow as tf
# import numpy as np
import matplotlib.pyplot as plt
import os

# Load MNIST Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Show Sample Images
plt.figure(figsize=(8,4))
for i in range(6):
    plt.subplot(2,3,i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title("Label: " + str(y_train[i]))
    plt.axis("off")
plt.show()

# Build Model
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train Model
history = model.fit(x_train, y_train, epochs=5,
                    validation_data=(x_test, y_test))

# Plot Accuracy Graph
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Test Accuracy')
plt.title("Accuracy Graph")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Save Normal Model
model.save("normal_mnist.h5")

# File Size Before
size_before = os.path.getsize("normal_mnist.h5") / 1024

# Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quant_model = converter.convert()

with open("quantized_mnist.tflite", "wb") as f:
    f.write(quant_model)

# File Size After
size_after = os.path.getsize("quantized_mnist.tflite") / 1024

# Bar Chart for Size Comparison
models = ['Normal', 'Quantized']
sizes = [size_before, size_after]

plt.bar(models, sizes)
plt.title("Model Size Comparison (KB)")
plt.ylabel("Size in KB")
plt.show()

print("Quantization Completed!")
print("Before:", round(size_before,2), "KB")
print("After :", round(size_after,2), "KB")

In [ ]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import matplotlib.pyplot as plt
import os

# Load Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Build Model
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Apply Pruning
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

model = prune_low_magnitude(
    model,
    pruning_schedule=tfmot.sparsity.keras.ConstantSparsity(
        target_sparsity=0.50,
        begin_step=0
    )
)

# Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train
history = model.fit(x_train, y_train,
                    epochs=5,
                    validation_data=(x_test, y_test),
                    verbose=1)

# Accuracy Graph
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Test Accuracy')
plt.title("Pruning Accuracy Graph")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Remove Wrapper
model = tfmot.sparsity.keras.strip_pruning(model)

# Save
model.save("pruned_mnist.h5")

# File Size
size = os.path.getsize("pruned_mnist.h5") / 1024

# Pie Chart
labels = ['Used Weights', 'Removed Weights']
values = [50, 50]

plt.pie(values, labels=labels, autopct='%1.1f%%')
plt.title("Pruning 50% Weights")
plt.show()

print("Pruning Completed!")
print("Pruned Model Size:", round(size,2), "KB")